<a href="https://colab.research.google.com/github/ayushi777lodhi-stack/GraphicalNeuralNetworks/blob/main/TGN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install skyfield
!pip install networkx
!pip install matplotlib
!pip install pandas
!pip install numpy
!pip install torch-geometric
!pip install scipy

In [ ]:
import networkx as nx
import numpy as np
from scipy.spatial import KDTree
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
from skyfield.api import load,EarthSatellite
import pandas as pd
from sklearn.metrics import (roc_auc_score,average_precision_score,precision_score,recall_score,f1_score)

In [ ]:
import torch
from torch_geometric.data import Data
import math
import torch.nn as nn
import torch.nn.functional as F
import random
from torch.utils.data import Dataset, DataLoader

In [ ]:
satellites=load.tle_file("starlink.tle")
print(f"loaded {len(satellites)} satellites")

In [ ]:
ts = load.timescale()

t = ts.now()

In [ ]:
t = ts.utc(2026, 7, 24, 12, 0, 0)

In [ ]:
import pandas as pd
dataset=[]
for sat in satellites:
    geocentric=sat.at(t)
    position=geocentric.position.km
    velocity=geocentric.velocity.km_per_s

    dataset.append([
        sat.name,
        t.utc_iso(),
        position[0],
        position[1],
        position[2],
        velocity[0],
        velocity[1],
        velocity[2]
    ])

df=pd.DataFrame(
    dataset,
    columns=[
        "name",
        "time",
        "x",
        "y",
        "z",
        "vx",
        "vy",
        "vz"
    ]
)

print(df)

In [ ]:
snapshots=[]
times=[
    ts.utc(2026,7,24,12,0),
    ts.utc(2026,7,24,12,5),
    ts.utc(2026,7,24,12,10),
    ts.utc(2026,7,24,12,15),
]
for t in times:
    rows=[]
    for sat in satellites:
        geocentric=sat.at(t)
        position=geocentric.position.km
        velocity=geocentric.velocity.km_per_s

        rows.append([
            sat.name,
            position[0],
            position[1],
            position[2],
            velocity[0],
            velocity[1],
            velocity[2]
        ])

    df=pd.DataFrame(
        rows,
        columns=[
            "name",
            "x",
            "y",
            "z",
            "vx",
            "vy",
            "vz"
        ]
    )

    snapshots.append(df)



In [ ]:
snapshots[0].head()

In [ ]:
snapshots[1].head()

In [ ]:
def build_graph(df, communication_radius=1000):

    G=nx.Graph()
    for _,row in df.iterrows():
        G.add_node(
            row["name"],
            pos=(row["x"], row["y"], row["z"]),
            velocity=(row["vx"], row["vy"], row["vz"])
        )

    positions=df[["x","y","z"]].values
    tree=KDTree(positions)
    for i,point in enumerate(positions):
        neighbors=tree.query_ball_point(point, communication_radius)

        for j in neighbors:
            if i>=j:
                continue
            distance=np.linalg.norm(positions[i] - positions[j])

            G.add_edge(
                df.iloc[i]["name"],
                df.iloc[j]["name"],
                distance=distance
            )

    return G

In [ ]:
def get_snapshot(satellites,t):
    rows=[]
    for sat in satellites:
        geocentric=sat.at(t)
        position=geocentric.position.km
        velocity=geocentric.velocity.km_per_s

        rows.append({
            "time":t,
            "name":sat.name,
            "x":position[0],
            "y":position[1],
            "z":position[2],
            "vx":velocity[0],
            "vy":velocity[1],
            "vz":velocity[2]
        })

    return pd.DataFrame(rows)

In [ ]:
start=datetime(2026,7,24,0,0)
times=[]

for i in range(100):
    current=start+timedelta(minutes=5*i)
    times.append(ts.utc(current.year,
            current.month,
            current.day,
            current.hour,
            current.minute,
            current.second))

In [ ]:
print(times)

In [ ]:
graphs=[]
for t in times:
    df=get_snapshot(satellites[:400], t)
    G=build_graph(df)
    graphs.append(G)

In [ ]:
print(df.head())

In [ ]:
for G in graphs:
    print(G)

In [ ]:
for G in graphs[:10]:
    fig=plt.figure(figsize=(10,10))
    ax=fig.add_subplot(111, projection="3d")

    for u,v in G.edges():
       p1=np.array(G.nodes[u]["pos"])
       p2=np.array(G.nodes[v]["pos"])

       ax.plot(
           [p1[0], p2[0]],
        [p1[1], p2[1]],
        [p1[2], p2[2]],
        color="black",
        linewidth=1
    )

    xs=[]
    ys=[]
    zs=[]

    for node in G.nodes():
        x,y,z=G.nodes[node]["pos"]

        xs.append(x)
        ys.append(y)
        zs.append(z)
        ax.scatter(xs, ys, zs, color="red", s=20)
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_zlabel("Z")
        plt.show()

In [ ]:
G.nodes

In [ ]:
node_to_idx={
    node:i
    for i,node in enumerate(G.nodes())
}

print(node_to_idx)

In [ ]:
def networkx_to_pyg(G):
    node_features=[]
    for node in G.nodes():
       x,y,z=G.nodes[node]["pos"]
       vx,vy,vz=G.nodes[node]["velocity"]
       node_features.append([x,y,z,vx,vy,vz])

    x=torch.tensor(node_features, dtype=torch.float)

    edges=[]
    for u,v in G.edges():
        edges.append([node_to_idx[u],node_to_idx[v]])
        edges.append([node_to_idx[v],node_to_idx[u]])

    edge_index=torch.tensor(edges,dtype=torch.long).t().contiguous()

    edge_attr=[]
    for u,v in G.edges():
        d=G.edges[u,v]["distance"]
        edge_attr.append([d])
        edge_attr.append([d])

    edge_attr=torch.tensor(edge_attr, dtype=torch.float)

    return Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr
    )

In [ ]:
for G in graphs:
  print(G)

In [ ]:
pyg_graphs=[]

for G in graphs:
    data=networkx_to_pyg(G)
    pyg_graphs.append(data)

In [ ]:
print(pyg_graphs)

In [ ]:
class SatelliteTemporalDataset(Dataset):
    def __init__(self,graphs,history):
        self.graphs=graphs
        self.history=history

    def __len__(self):
        return len(self.graphs)-self.history

    def __getitem__(self,idx):
        input_sequence=self.graphs[idx:idx+self.history]
        target_graph=self.graphs[idx+self.history]

        return input_sequence,target_graph

In [ ]:
history=4
dataset=SatelliteTemporalDataset(
    pyg_graphs,
    history=history
)

In [ ]:
print(len(dataset))

In [ ]:
def collate_fn(batch):
    histories=[]
    targets=[]

    for history,target in batch:
        histories.append(history)
        targets.append(target)

    return histories, targets

In [ ]:
loader=DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

In [ ]:
for histories,targets in loader:
    print(len(histories))
    print(len(targets))
    break

In [ ]:
from torch_geometric.utils import softmax
from torch_geometric.utils import scatter


class StructuralAttention(nn.Module):
    def __init__(self,in_dim,hidden_dim,num_heads=8,dropout=0.1):
        super().__init__()
        assert hidden_dim%num_heads==0

        self.hidden_dim=hidden_dim
        self.num_heads=num_heads
        self.head_dim=hidden_dim//num_heads
        self.W=nn.Linear(in_dim,hidden_dim,bias=False)
        self.attn=nn.Parameter(torch.empty(num_heads,2*self.head_dim))
        nn.init.uniform_(self.attn,-0.1,0.1)
        self.dropout=nn.Dropout(dropout)

    def forward(self,x,edge_index):
        src,dst=edge_index
        N=x.size(0)
        Wh=self.W(x)
        Wh=Wh.view(N,self.num_heads,self.head_dim)
        Wh_src=Wh[src]
        Wh_dst=Wh[dst]
        edge_features=torch.cat([Wh_src, Wh_dst],dim=-1)
        e=(edge_features*self.attn.unsqueeze(0)).sum(dim=-1)
        e=F.leaky_relu(e,0.2)
        alpha=softmax(e,dst)
        alpha=self.dropout(alpha)
        out=scatter(alpha.unsqueeze(-1)*Wh_src,dst,dim=0,dim_size=N,reduce="sum")
        out=out.reshape(N,self.hidden_dim)

        return F.elu(out)

In [ ]:
class StructuralBlock(nn.Module):
    def __init__(self,in_dim,hidden_dim,num_layers):
        super().__init__()

        self.layers=nn.ModuleList([
    StructuralAttention(
        in_dim if i==0 else hidden_dim,
        hidden_dim,
        num_heads=8)
    for i in range(num_layers)
    ])

    def forward(self, x, edge_index):
        for layer in self.layers:
            x=layer(x, edge_index)

        return x

In [ ]:
structural=StructuralBlock(
    in_dim=6,
    hidden_dim=64,
    num_layers=2
)

In [ ]:
class MultiHeadTemporalAttention(nn.Module):
    def __init__(self,hidden_dim,num_heads=8,dropout=0.1):
        super().__init__()
        assert hidden_dim%num_heads==0

        self.hidden_dim=hidden_dim
        self.num_heads=num_heads
        self.head_dim=hidden_dim//num_heads

        self.query=nn.Linear(hidden_dim,hidden_dim,bias=False)
        self.key=nn.Linear(hidden_dim,hidden_dim,bias=False)
        self.value=nn.Linear(hidden_dim,hidden_dim,bias=False)

        self.out_proj=nn.Linear(hidden_dim,hidden_dim)
        self.dropout=nn.Dropout(dropout)

    def forward(self,x):
        x=x.permute(1,0,2).contiguous()
        N,T,_ =x.shape
        Q=self.query(x)
        K=self.key(x)
        V=self.value(x)


        Q=Q.reshape(N,T,self.num_heads,self.head_dim)
        K=K.reshape(N,T,self.num_heads,self.head_dim)
        V=V.reshape(N,T,self.num_heads,self.head_dim)

        Q=Q.transpose(1,2)
        K=K.transpose(1,2)
        V=V.transpose(1,2)
        scores=torch.matmul(Q,K.transpose(-2,-1))
        scores=scores/math.sqrt(self.head_dim)

        mask=torch.triu(torch.ones(T,T,device=x.device),diagonal=1).bool()
        scores=scores.masked_fill(mask,float("-inf"))
        attention=F.softmax(scores,dim=-1)
        attention=self.dropout(attention)
        out=torch.matmul(attention,V)
        out=out.transpose(1,2).contiguous()
        out=out.view(N,T,self.hidden_dim)
        out=self.out_proj(out)
        out=out.permute(1,0,2)

        return out

In [ ]:
class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim, dropout=0.1):
        super().__init__()

        self.hidden_dim=hidden_dim
        self.Wq=nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.Wk=nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.Wv=nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.dropout=nn.Dropout(dropout)

    def forward(self, x):
        x=x.permute(1,0,2)
        Q=self.Wq(x)
        K=self.Wk(x)
        V=self.Wv(x)
        scores = torch.matmul(Q,K.transpose(-2, -1))
        scores=scores/math.sqrt(self.hidden_dim)
        T=scores.size(-1)
        mask=torch.triu(torch.ones(T,T,device=x.device),diagonal=1).bool()
        scores=scores.masked_fill(mask,float("-inf"))
        attention=F.softmax(scores,dim=-1)
        attention=self.dropout(attention)
        out=torch.matmul(attention,V)
        out=out.permute(1,0,2)

        return out

In [ ]:
class PositionEmbedding(nn.Module):
    def __init__(self,num_snapshots,hidden_dim):
        super().__init__()
        self.position=nn.Parameter(torch.randn(num_snapshots, hidden_dim))

    def forward(self,x):
        return x+self.position[:, None, :]

In [ ]:
class FeedForward(nn.Module):
    def __init__(self,hidden_dim,dropout=0.1):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(hidden_dim,hidden_dim*4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim*4,hidden_dim)
         )

    def forward(self,x):
        return self.net(x)

In [ ]:
class TemporalBlock(nn.Module):
    def __init__(self,hidden_dim,num_layers,num_snapshots):
        super().__init__()
        self.position_embedding=nn.Parameter(torch.randn(num_snapshots, hidden_dim))
        self.attention=nn.ModuleList([
            MultiHeadTemporalAttention(
            hidden_dim,
            num_heads=8,
            dropout=0.1)
            for _ in range(num_layers)
        ])

        self.feed_forward=nn.ModuleList([
            FeedForward(hidden_dim)
            for _ in range(num_layers)
        ])

        self.norm1=nn.ModuleList([
            nn.LayerNorm(hidden_dim)
            for _ in range(num_layers)
        ])

        self.norm2=nn.ModuleList([
            nn.LayerNorm(hidden_dim)
            for _ in range(num_layers)
        ])

    def forward(self,H):
      H=H+self.position_embedding[:, None, :]

      for attn, ff, norm1, norm2 in zip(self.attention,self.feed_forward,self.norm1,self.norm2):
        H=norm1(H+attn(H))
        H=norm2(H+ff(H))

      return H

In [ ]:
class DySAT(nn.Module):
    def __init__(self,input_dim,hidden_dim,structural_layers,temporal_layers,num_snapshots):
        super().__init__()
        self.structural=StructuralBlock(
            in_dim=input_dim,
            hidden_dim=hidden_dim,
            num_layers=structural_layers
        )

        self.temporal=TemporalBlock(
            hidden_dim=hidden_dim,
            num_layers=temporal_layers,
            num_snapshots=num_snapshots
        )

    def forward(self, graphs):
        structural_embeddings=[]
        for graph in graphs:
            h=self.structural(graph.x,graph.edge_index)
            structural_embeddings.append(h)

        H=torch.stack(structural_embeddings,dim=0)
        H=self.temporal(H)

        return H

In [ ]:
print(graphs[0])

In [ ]:
print(pyg_graphs)

In [ ]:
print(len(pyg_graphs))

In [ ]:
history=4
device="cuda" if torch.cuda.is_available() else "cpu"
model=DySAT(
    input_dim=6,
    hidden_dim=64,
    structural_layers=2,
    temporal_layers=2,
    num_snapshots=history)
model=model.to(device)
history_graphs=[g.to(device) for g in pyg_graphs[:4]]
embeddings=model(history_graphs)


In [ ]:
class LinkPredictor(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.predictor=nn.Sequential(
            nn.Linear(hidden_dim*2,hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim,1))

    def forward(self, embeddings,edge_index):
        src,dst=edge_index
        edge_embeddings=torch.cat([embeddings[src],embeddings[dst]],dim=1)
        logits=self.predictor(edge_embeddings)

        return logits.squeeze(-1)

In [ ]:
split=int(0.8*len(pyg_graphs))
train_graphs=pyg_graphs[:split]
test_graphs=pyg_graphs[split:]

In [ ]:
len(pyg_graphs)

In [ ]:
print(train_graphs)
print(pyg_graphs)
print(test_graphs)

In [ ]:
HISTORY=4
train_dataset=SatelliteTemporalDataset(
    train_graphs,
    history=HISTORY
)

test_dataset=SatelliteTemporalDataset(
    test_graphs,
    history=HISTORY
)

In [ ]:
def temporal_collate(batch):
    histories=[item[0] for item in batch]
    targets=[item[1] for item in batch]

    return histories,targets

In [ ]:
train_loader=DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=temporal_collate
)

test_loader=DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=temporal_collate
)

In [ ]:
predictor=LinkPredictor(hidden_dim=64)

In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=model.to(device)
predictor=predictor.to(device)
optimizer=torch.optim.Adam(list(model.parameters())+list(predictor.parameters()),lr=1e-3)
criterion=nn.BCEWithLogitsLoss()

In [ ]:
from torch_geometric.utils import negative_sampling
loss_history=[]
epochs=50

for epoch in range(epochs):

    model.train()
    predictor.train()

    total_loss=0
    for histories,targets in train_loader:
        batch_loss=0
        optimizer.zero_grad()

        for history,target_graph in zip(histories, targets):
            history=[g.to(device) for g in history]
            target_graph=target_graph.to(device)
            H=model(history)
            embeddings=H[-1]

            positive_edges=target_graph.edge_index
            negative_edges=negative_sampling(
                edge_index=positive_edges,
                num_nodes=target_graph.num_nodes,
                num_neg_samples=positive_edges.size(1),
                method="sparse"
            ).to(device)
            positive_edges=positive_edges.to(device)
            positive_logits=predictor(embeddings,positive_edges)
            negative_logits=predictor(embeddings,negative_edges)
            logits=torch.cat([positive_logits,negative_logits])
            labels=torch.cat([torch.ones_like(positive_logits),torch.zeros_like(negative_logits)])

            loss=criterion(logits,labels)
            batch_loss+=loss

        batch_loss.backward()
        optimizer.step()
        total_loss+=batch_loss.item()

    avg_loss=total_loss/len(train_loader)
    loss_history.append(avg_loss)
    print(
        f"Epoch {epoch+1:03d} loss={avg_loss:.4f}"
    )

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.grid(True)
plt.show()

In [ ]:
torch.save({
    "dysat":model.state_dict(),
     "predictor":predictor.state_dict()}
    ,"dysat_link_prediction.pth",)

In [ ]:
import itertools
from matplotlib.lines import Line2D
model.eval()
predictor.eval()

history,target_graph=test_dataset[0]
history=[g.to(device) for g in history]
target_graph=target_graph.to(device)

with torch.no_grad():
    H=model(history)

embeddings=H[-1]
num_nodes=target_graph.num_nodes

all_edges=torch.tensor(list(itertools.combinations(range(num_nodes),2)),dtype=torch.long).t().to(device)

with torch.no_grad():
    logits=predictor(embeddings,all_edges)

prob=torch.sigmoid(logits)
real_edges=target_graph.edge_index
K=real_edges.size(1)
topk=torch.topk(prob,K)
pred_edges=all_edges[:,topk.indices]
real=set(tuple(sorted(edge)) for edge in real_edges.t().cpu().tolist())
pred=set(tuple(sorted(edge)) for edge in pred_edges.t().cpu().tolist())

tp=real&pred
fp=pred-real
fn=real-pred

print(f"Real edges:{len(real)}")
print(f"Predicted edges:{len(pred)}")
print(f"TP:{len(tp)}")
print(f"FP:{len(fp)}")
print(f"FN:{len(fn)}")
print(f"Probability range:{prob.min():.4f} - {prob.max():.4f}")
print(f"Mean probability:{prob.mean():.4f}")


G=nx.Graph()
G.add_nodes_from(range(num_nodes))
G.add_edges_from(real)
G.add_edges_from(pred)
pos=nx.spring_layout(G,seed=42)
plt.figure(figsize=(10,10))

nx.draw_networkx_nodes(G,pos,node_size=10,node_color="blue")
nx.draw_networkx_edges(G,pos,edgelist=list(fn),edge_color="blue",style="dashed",width=2)
nx.draw_networkx_edges(G,pos,edgelist=list(fp),edge_color="red",width=2)
nx.draw_networkx_edges(G,pos,edgelist=list(tp),edge_color="green",width=2)

legend=[
    Line2D(
        [0],
        [0],
        color="green",
        lw=3,
        label="Correct Prediction"
    ),

    Line2D(
        [0],
        [0],
        color="red",
        lw=3,
        label="False Positive"
    ),

    Line2D(
        [0],
        [0],
        color="blue",
        lw=3,
        linestyle="--",
        label="False Negative"
    )]

plt.legend(handles=legend)
plt.title(
    f"DySAT Link Prediction\n"
    f"TP={len(tp)}   FP={len(fp)}   FN={len(fn)}")
plt.axis("off")
plt.show()

In [ ]:
model.eval()
predictor.eval()
all_labels=[]
all_scores=[]

with torch.no_grad():
    for histories,targets in test_loader:
        for history,target_graph in zip(histories, targets):

            history=[g.to(device) for g in history]
            target_graph=target_graph.to(device)
            H=model(history)
            embeddings=H[-1]

            positive_edges=target_graph.edge_index
            negative_edges=negative_sampling(
                edge_index=positive_edges,
                num_nodes=target_graph.num_nodes,
                num_neg_samples=positive_edges.size(1),
                method="sparse"
            ).to(device)
            positive_edges=positive_edges.to(device)
            positive_logits=predictor(embeddings,positive_edges)
            negative_logits=predictor(embeddings,negative_edges)
            logits=torch.cat([positive_logits,negative_logits])
            labels=torch.cat([torch.ones_like(positive_logits),torch.zeros_like(negative_logits)])
            scores=torch.sigmoid(logits)
            all_scores.extend(scores.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

In [ ]:
all_scores=np.array(all_scores)
all_labels=np.array(all_labels)
predictions=(all_scores > 0.5).astype(int)

In [ ]:
roc=roc_auc_score(all_labels,all_scores)
ap=average_precision_score(all_labels,all_scores)
precision=precision_score(all_labels,predictions)
recall=recall_score(all_labels,predictions)
f1=f1_score(all_labels,predictions)

print(f"ROC-AUC:{roc:.4f}")
print(f"AP:{ap:.4f}")
print(f"Precision:{precision:.4f}")
print(f"Recall:{recall:.4f}")
print(f"F1 Score:{f1:.4f}")

In [ ]:
class CollisionDetector:
    def __init__(self,satellites,ts,search_radius=200,collision_threshold=5):

        self.satellites=satellites
        self.ts=ts
        self.search_radius=search_radius
        self.collision_threshold=collision_threshold


    def propagate_positions(self, time):
        positions=[]

        for sat in self.satellites:
            geocentric=sat.at(time)
            positions.append(geocentric.position.km)

        return np.array(positions)

    def candidate_pairs(self,positions):
        tree=KDTree(positions)
        return list(tree.query_pairs(self.search_radius))

    def detect(self,start_time,horizon_minutes=10,step_seconds=30):
        alerts=[]
        positions=self.propagate_positions(start_time)
        candidates=self.candidate_pairs(positions)

        for sat1,sat2 in candidates:
            minimum_distance=np.inf
            best_time=None

            for dt in range(0,horizon_minutes*60+1,step_seconds):
                future=self.ts.utc(start_time.utc_datetime()+timedelta(seconds=dt))
                p1=self.satellites[sat1].at(future).position.km
                p2=self.satellites[sat2].at(future).position.km
                distance=np.linalg.norm(p1-p2)

                if distance<minimum_distance:
                    minimum_distance=distance
                    best_time=future

            if minimum_distance<self.collision_threshold:
                alerts.append({
                    "satellite_1":self.satellites[sat1].name,
                    "satellite_2":self.satellites[sat2].name,
                    "minimum_distance_km":minimum_distance,
                    "time":best_time.utc_iso()})

        return alerts

In [ ]:
detector=CollisionDetector(satellites,ts,search_radius=200,collision_threshold=5)
alerts=detector.detect(t,horizon_minutes=15,step_seconds=30)

In [ ]:
for alert in alerts:
    print(f"sat 1: {alert['satellite_1']}")
    print(f"sat 2: {alert['satellite_2']}")
    print(f"min dist: {alert['minimum_distance_km']:.2f} km")
    print(f"time of closest approach: {alert['time']}")

In [ ]:
def compute_network_metrics(G):
    metrics={}
    metrics["num_nodes"]=G.number_of_nodes()
    metrics["num_edges"]=G.number_of_edges()
    metrics["connected_components"]=nx.number_connected_components(G)
    largest_cc =max(nx.connected_components(G),key=len)
    metrics["largest_component"]=len(largest_cc)
    metrics["density"]=nx.density(G)
    metrics["average_degree"]=(sum(dict(G.degree()).values())/G.number_of_nodes())
    metrics["clustering"]=nx.average_clustering(G)
    metrics["global_efficiency"]=nx.global_efficiency(G)
    largest_graph=G.subgraph(largest_cc)

    if largest_graph.number_of_nodes()>1:
        metrics["average_shortest_path"]=(nx.average_shortest_path_length(largest_graph))
        metrics["diameter"]=(nx.diameter(largest_graph))

    else:
        metrics["average_shortest_path"]=0
        metrics["diameter"]=0

    return metrics

In [ ]:
baseline_metrics=compute_network_metrics(G)
pd.DataFrame([baseline_metrics])

In [ ]:
class RoutePlanner:

    def __init__(self, graph):
        self.graph=graph.copy()

    def apply_collision_penalties(self,collision_alerts,penalty=1e9):

        G=self.graph.copy()
        risky_nodes=set()

        for alert in collision_alerts:
            risky_nodes.add(alert["satellite_1"])
            risky_nodes.add(alert["satellite_2"])

        for node in risky_nodes:
            if node not in G:
                continue

            for neighbor in G.neighbors(node):
                G[node][neighbor]["weight"]+=penalty

        return G



    def shortest_path(self,source,destination,collision_alerts=None):

        if collision_alerts is None:
            graph=self.graph
        else:
            graph=self.apply_collision_penalties(collision_alerts)

        path=nx.shortest_path(graph,source=source,target=destination,weight="weight")
        distance=nx.shortest_path_length(graph,source=source,target=destination,weight="weight")

        return {
            "path":path,
            "total_cost":distance,
            "num_hops":len(path)-1}





In [ ]:
largest_component=list(max(nx.connected_components(G),key=len))
print(len(largest_component))

source=largest_component[0]
destination=largest_component[50]

planner=RoutePlanner(G)
route=planner.shortest_path(source=source,destination=destination,collision_alerts=alerts)
print(route)

In [ ]:
def draw_route(G,route):
    pos=nx.spring_layout(G,seed=42)
    plt.figure(figsize=(10,8))
    nx.draw_networkx_edges(G,pos,alpha=0.3)
    nx.draw_networkx_nodes(G,pos,node_size=10)
    path_edges=list(zip(route[:-1],route[1:]))
    nx.draw_networkx_edges(G,pos,edgelist=path_edges,width=3)
    plt.show()

In [ ]:
draw_route(G,route["path"])